# 02 Correlation & Regression Analysis

Analyze the relationships between Nikkei 225 and macro/micro indicators.

| Section | Method |
|---------|--------|
| 1 | Data loading & preprocessing |
| 2 | Pearson correlation heatmap |
| 3 | Lag correlation (CCF) |
| 4 | Rolling correlation |
| 5 | Mutual information |
| 6 | Multiple regression (OLS) + VIF |
| 7 | Granger causality test |
| 8 | Cointegration test (Engle-Granger / Johansen) |
| 9 | VAR / VECM |
| 10 | PCA |
| 11 | Summary & feature ranking |

## 0. Environment Setup (Google Colab)

In [ ]:
import os, sys

REPO = '/content/Nikkei_Analysis'

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Clone if first time, otherwise pull latest changes
    if not os.path.exists(REPO):
        !git clone https://github.com/Takumi-Itokawa-Finance/Nikkei_Analysis.git {REPO}
    else:
        !git -C {REPO} pull

    # Set working directory to repo root
    os.chdir(REPO)

    # Mount Google Drive for data persistence
    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_DATA = '/content/drive/MyDrive/Nikkei_Analysis/data'
    os.makedirs(DRIVE_DATA, exist_ok=True)
    if not os.path.exists(f'{REPO}/data'):
        os.symlink(DRIVE_DATA, f'{REPO}/data')

    !pip install -q yfinance pandas-ta pandas-datareader

    # Ensure repo root is on Python path so src/ is importable
    if REPO not in sys.path:
        sys.path.insert(0, REPO)

    print(f'Colab setup complete.  CWD={os.getcwd()}  path[0]={sys.path[0]}')
else:
    print('Running in local environment.')

## 1. Data Loading & Preprocessing

In [ ]:
import os, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

# Fallback: ensure src/ is importable even if cell 0 was skipped
_repo = '/content/Nikkei_Analysis'
if os.path.isdir(_repo) and _repo not in sys.path:
    sys.path.insert(0, _repo)

from src.data.fetcher import fetch_all

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid', palette='muted')

OUTPUT_DIR = 'output/figures'
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# Load all Close prices
raw = fetch_all(period='5y', interval='1d')

# Forward-fill gaps (weekends, holidays) then drop leading NaNs
prices = raw.ffill().dropna()

# Log returns (stationary, used for most analyses)
returns = np.log(prices / prices.shift(1)).dropna()

print(f'Price levels  : {prices.shape}  {prices.index[0].date()} to {prices.index[-1].date()}')
print(f'Log returns   : {returns.shape}')
print(f'Columns       : {returns.columns.tolist()}')

In [ ]:
# Target and feature series
TARGET = 'Nikkei225'
features = [c for c in returns.columns if c != TARGET]

print(f'Target  : {TARGET}')
print(f'Features: {len(features)}')
display(returns.describe().T)

## 2. Pearson Correlation Heatmap

Computed on log returns (not price levels) to avoid spurious correlation from shared trends.

In [ ]:
corr = returns.corr()

# Sort features by absolute correlation with Nikkei 225
nikkei_corr = corr[TARGET].drop(TARGET).sort_values(key=abs, ascending=False)
print('Correlation with Nikkei 225 (log returns):')
display(nikkei_corr.to_frame('pearson_r'))

In [ ]:
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            linewidths=0.5, ax=ax, annot_kws={'size': 8})
ax.set_title('Pearson Correlation Matrix (log returns)')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/02_pearson_heatmap.png', dpi=150)
plt.show()

In [ ]:
# Bar chart: correlation of each feature with Nikkei 225
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['steelblue' if v >= 0 else 'tomato' for v in nikkei_corr]
nikkei_corr.plot(kind='barh', ax=ax, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Pearson Correlation with Nikkei 225')
ax.set_xlabel('Correlation coefficient')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/02_pearson_bar.png', dpi=150)
plt.show()

## 3. Lag Correlation (Cross-Correlation Function)

Identifies whether indicators **lead** Nikkei 225 (negative lag = indicator moves first).

In [ ]:
def cross_corr(target: pd.Series, feature: pd.Series, max_lag: int = 10) -> pd.Series:
    """Compute cross-correlation for lags -max_lag to +max_lag."""
    lags = range(-max_lag, max_lag + 1)
    return pd.Series(
        {lag: target.corr(feature.shift(lag)) for lag in lags},
        name=feature.name
    )

MAX_LAG = 10
ccf_results = {f: cross_corr(returns[TARGET], returns[f], MAX_LAG) for f in features}

# Show lag with peak absolute correlation for each feature
peak_lags = pd.DataFrame([
    {'feature': f,
     'best_lag': int(s.abs().idxmax()),
     'peak_corr': round(s[s.abs().idxmax()], 4)}
    for f, s in ccf_results.items()
]).sort_values('peak_corr', key=abs, ascending=False)

display(peak_lags)

In [ ]:
# Plot CCF for top 6 features by absolute peak correlation
top6 = peak_lags.head(6)['feature'].tolist()
fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharey=False)
lags = range(-MAX_LAG, MAX_LAG + 1)

for ax, feat in zip(axes.flat, top6):
    s = ccf_results[feat]
    colors = ['steelblue' if v >= 0 else 'tomato' for v in s]
    ax.bar(lags, s.values, color=colors)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.axvline(0, color='grey', linewidth=0.8, linestyle='--')
    ax.set_title(feat)
    ax.set_xlabel('Lag (days)')
    ax.set_ylabel('Correlation')

fig.suptitle('Cross-Correlation with Nikkei 225 (negative lag = indicator leads)', y=1.01)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/02_ccf_top6.png', dpi=150)
plt.show()

## 4. Rolling Correlation

Correlations in financial markets are **not stable over time** — they shift during crises, policy changes, and regime transitions.
A 60-day rolling window reveals these structural changes.

In [ ]:
WINDOW = 60  # trading days (~3 months)

rolling_corr = pd.DataFrame({
    f: returns[TARGET].rolling(WINDOW).corr(returns[f])
    for f in features
})

# Select top features by mean absolute rolling correlation
mean_abs = rolling_corr.abs().mean().sort_values(ascending=False)
top_rolling = mean_abs.head(6).index.tolist()
print('Top features by mean absolute rolling correlation:')
display(mean_abs.head(10).to_frame('mean_abs_corr'))

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(15, 12), sharex=True)

for ax, feat in zip(axes.flat, top_rolling):
    s = rolling_corr[feat].dropna()
    ax.plot(s.index, s.values, linewidth=1)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.fill_between(s.index, s.values, 0,
                    where=(s.values >= 0), alpha=0.2, color='steelblue')
    ax.fill_between(s.index, s.values, 0,
                    where=(s.values < 0), alpha=0.2, color='tomato')
    ax.set_title(f'{feat} (60d rolling)')
    ax.set_ylabel('Correlation')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

fig.suptitle('Rolling Correlation with Nikkei 225 (60-day window)', y=1.01)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/02_rolling_corr.png', dpi=150)
plt.show()

## 5. Mutual Information

Pearson correlation only captures **linear** relationships.
Mutual information (MI) detects any statistical dependency, including non-linear ones.
Higher MI = more information shared with the target.

In [ ]:
from sklearn.feature_selection import mutual_info_regression
from sklearn.preprocessing import StandardScaler

X = returns[features].values
y = returns[TARGET].values

mi_scores = mutual_info_regression(X, y, random_state=42)
mi_df = pd.DataFrame({'feature': features, 'MI': mi_scores})
mi_df = mi_df.sort_values('MI', ascending=False)

display(mi_df)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(mi_df['feature'], mi_df['MI'], color='steelblue')
ax.set_title('Mutual Information with Nikkei 225 (log returns)')
ax.set_xlabel('Mutual Information score')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/02_mutual_info.png', dpi=150)
plt.show()

In [ ]:
# Compare Pearson absolute value vs MI rank
comparison = pd.DataFrame({
    'pearson_abs': nikkei_corr.abs(),
    'MI': mi_df.set_index('feature')['MI'],
}).sort_values('MI', ascending=False)

comparison['pearson_rank'] = comparison['pearson_abs'].rank(ascending=False).astype(int)
comparison['MI_rank']      = comparison['MI'].rank(ascending=False).astype(int)
comparison['rank_diff']    = (comparison['pearson_rank'] - comparison['MI_rank']).abs()

print('Features with large rank discrepancy (possible non-linear relationship):')
display(comparison.sort_values('rank_diff', ascending=False))

## 6. Multiple Regression (OLS) + VIF

Quantify the marginal contribution of each indicator after controlling for others,
and check for multicollinearity via Variance Inflation Factor (VIF).

In [ ]:
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

X_ols = sm.add_constant(returns[features])
y_ols = returns[TARGET]

ols_model = sm.OLS(y_ols, X_ols).fit()
print(ols_model.summary())

In [ ]:
# VIF — rule of thumb: VIF > 10 indicates severe multicollinearity
vif_data = pd.DataFrame({
    'feature': X_ols.columns,
    'VIF': [variance_inflation_factor(X_ols.values, i) for i in range(X_ols.shape[1])]
}).query('feature != "const"').sort_values('VIF', ascending=False)

print('VIF (>10 = high multicollinearity, consider dropping):')
display(vif_data)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['tomato' if v > 10 else 'steelblue' for v in vif_data['VIF']]
ax.barh(vif_data['feature'], vif_data['VIF'], color=colors)
ax.axvline(10, color='red', linestyle='--', linewidth=1, label='VIF=10 threshold')
ax.set_title('Variance Inflation Factor (VIF)')
ax.set_xlabel('VIF')
ax.legend()
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/02_vif.png', dpi=150)
plt.show()

In [ ]:
# OLS coefficient plot (significant predictors only, p < 0.05)
coef_df = pd.DataFrame({
    'coef':  ols_model.params,
    'pval':  ols_model.pvalues,
    'ci_lo': ols_model.conf_int()[0],
    'ci_hi': ols_model.conf_int()[1],
}).drop('const').sort_values('coef')

sig = coef_df[coef_df['pval'] < 0.05]
print(f'Significant predictors (p < 0.05): {len(sig)} / {len(coef_df)}')
display(sig)

In [ ]:
if len(sig) > 0:
    fig, ax = plt.subplots(figsize=(8, max(4, len(sig) * 0.5)))
    ax.barh(sig.index, sig['coef'],
            xerr=[sig['coef'] - sig['ci_lo'], sig['ci_hi'] - sig['coef']],
            color=['steelblue' if v >= 0 else 'tomato' for v in sig['coef']],
            capsize=4)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title('OLS Coefficients — Significant Predictors (p < 0.05)')
    ax.set_xlabel('Coefficient')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/02_ols_coef.png', dpi=150)
    plt.show()

## 7. Granger Causality Test

Tests whether past values of a feature statistically improve the prediction of Nikkei 225 returns.
Note: 'causality' here is statistical predictability, not true economic causation.

In [ ]:
from statsmodels.tsa.stattools import grangercausalitytests

MAX_LAG_GC = 5
gc_results = []

for feat in features:
    data = returns[[TARGET, feat]].dropna()
    try:
        res = grangercausalitytests(data, maxlag=MAX_LAG_GC, verbose=False)
        # Use minimum p-value across lags (F-test)
        min_pval = min(res[lag][0]['ssr_ftest'][1] for lag in range(1, MAX_LAG_GC + 1))
        best_lag = min(res, key=lambda lag: res[lag][0]['ssr_ftest'][1])
        gc_results.append({
            'feature':  feat,
            'min_pval': round(min_pval, 4),
            'best_lag': best_lag,
            'granger':  min_pval < 0.05,
        })
    except Exception as e:
        gc_results.append({'feature': feat, 'min_pval': None,
                           'best_lag': None, 'granger': None})

gc_df = pd.DataFrame(gc_results).sort_values('min_pval')
print(f"Granger causal (p < 0.05): {gc_df['granger'].sum()} / {len(gc_df)}")
display(gc_df)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
gc_plot = gc_df.dropna(subset=['min_pval'])
colors = ['tomato' if v < 0.05 else 'steelblue' for v in gc_plot['min_pval']]
ax.barh(gc_plot['feature'], gc_plot['min_pval'], color=colors)
ax.axvline(0.05, color='red', linestyle='--', linewidth=1, label='p=0.05')
ax.set_title('Granger Causality — Min p-value (lags 1-5)')
ax.set_xlabel('Min p-value (red = Granger causal)')
ax.legend()
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/02_granger.png', dpi=150)
plt.show()

## 8. Cointegration Test

Tests whether Nikkei 225 and each indicator share a long-run equilibrium (tested on **price levels**).
- Engle-Granger: pairwise test
- Johansen: multivariate test on the full feature set

If cointegration is found → use **VECM** instead of VAR.

In [ ]:
from statsmodels.tsa.stattools import coint

coint_results = []
for feat in features:
    data = prices[[TARGET, feat]].dropna()
    try:
        stat, pval, _ = coint(data[TARGET], data[feat])
        coint_results.append({
            'feature':       feat,
            'test_stat':     round(stat, 4),
            'pval':          round(pval, 4),
            'cointegrated':  pval < 0.05,
        })
    except Exception:
        coint_results.append({'feature': feat, 'test_stat': None,
                              'pval': None, 'cointegrated': None})

coint_df = pd.DataFrame(coint_results).sort_values('pval')
print(f"Cointegrated pairs (p < 0.05): {coint_df['cointegrated'].sum()} / {len(coint_df)}")
display(coint_df)

In [ ]:
# Johansen cointegration test (multivariate)
from statsmodels.tsa.vector_ar.vecm import select_coint_rank

prices_clean = prices.dropna()
johansen = select_coint_rank(prices_clean, det_order=0, k_ar_diff=1, method='trace')
print('Johansen Cointegration Test (trace):')
print(f'  Cointegration rank: {johansen.rank}')
print(f'  Conclusion: {"Use VECM" if johansen.rank > 0 else "Use VAR on returns"}')

## 9. VAR / VECM

- **VAR** (on log returns): if no cointegration
- **VECM** (on price levels): if cointegration rank > 0

We fit both for reference and use forecast accuracy to guide the final choice.

In [ ]:
from statsmodels.tsa.vector_ar.var_model import VAR

# Use top features by Granger causality (p < 0.05) to keep model tractable
granger_feats = gc_df[gc_df['granger'] == True]['feature'].tolist()
if len(granger_feats) == 0:
    granger_feats = gc_df.head(5)['feature'].tolist()  # fallback: top 5

var_cols = [TARGET] + granger_feats
var_data = returns[var_cols].dropna()

# Select optimal lag order
var_selector = VAR(var_data)
lag_order = var_selector.select_order(maxlags=10)
best_lag = lag_order.aic
print(f'VAR lag order selected by AIC: {best_lag}')
print(lag_order.summary())

In [ ]:
# Fit VAR
var_model = var_selector.fit(best_lag)
print(var_model.summary())

In [ ]:
# Forecast next 5 days
forecast = var_model.forecast(var_data.values[-best_lag:], steps=5)
forecast_df = pd.DataFrame(forecast, columns=var_cols)
print('VAR 5-day forecast (log returns):')
display(forecast_df[[TARGET]])

In [ ]:
# Impulse response — how does a shock to SP500 propagate to Nikkei?
irf = var_model.irf(periods=10)
irf.plot(impulse='SP500', response=TARGET, orth=True,
         subplot_params={'figsize': (10, 4)})
plt.suptitle('Impulse Response: SP500 shock -> Nikkei 225')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/02_irf_sp500_nikkei.png', dpi=150)
plt.show()

In [ ]:
# VECM (only if Johansen rank > 0)
if johansen.rank > 0:
    from statsmodels.tsa.vector_ar.vecm import VECM

    vecm_data = prices[var_cols].dropna()
    vecm_model = VECM(vecm_data, k_ar_diff=best_lag, coint_rank=johansen.rank,
                      deterministic='ci').fit()
    print(vecm_model.summary())
else:
    print('No cointegration detected — VAR on returns is appropriate.')

## 10. PCA

Reduce dimensionality of the feature set and identify the main drivers of variance.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

X_scaled = StandardScaler().fit_transform(returns[features].dropna())

pca = PCA()
pca.fit(X_scaled)

explained = pca.explained_variance_ratio_
cumulative = explained.cumsum()

n90 = int(np.argmax(cumulative >= 0.90)) + 1
print(f'Components to explain 90% variance: {n90}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scree plot
axes[0].bar(range(1, len(explained) + 1), explained, color='steelblue')
axes[0].set_title('PCA Scree Plot')
axes[0].set_xlabel('Component')
axes[0].set_ylabel('Explained variance ratio')

# Cumulative variance
axes[1].plot(range(1, len(cumulative) + 1), cumulative, marker='o', color='steelblue')
axes[1].axhline(0.90, color='red', linestyle='--', label='90% threshold')
axes[1].axvline(n90, color='grey', linestyle='--')
axes[1].set_title('Cumulative Explained Variance')
axes[1].set_xlabel('Number of components')
axes[1].set_ylabel('Cumulative variance')
axes[1].legend()

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/02_pca_variance.png', dpi=150)
plt.show()

In [ ]:
# Loadings heatmap for first 5 PCs
n_show = min(5, len(features))
loadings = pd.DataFrame(
    pca.components_[:n_show].T,
    index=features,
    columns=[f'PC{i+1}' for i in range(n_show)]
)

fig, ax = plt.subplots(figsize=(8, max(6, len(features) * 0.4)))
sns.heatmap(loadings, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, linewidths=0.5, ax=ax)
ax.set_title('PCA Loadings (first 5 components)')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/02_pca_loadings.png', dpi=150)
plt.show()

## 11. Summary & Feature Ranking

Aggregate evidence from all analyses into a single ranking table.

In [ ]:
ranking = pd.DataFrame({'feature': features}).set_index('feature')

# Pearson absolute correlation (higher = better)
ranking['pearson_abs']   = nikkei_corr.abs()

# Best lag CCF
ranking['ccf_peak']      = peak_lags.set_index('feature')['peak_corr'].abs()
ranking['ccf_best_lag']  = peak_lags.set_index('feature')['best_lag']

# Rolling corr mean abs
ranking['rolling_mean']  = mean_abs

# Mutual information
ranking['MI']            = mi_df.set_index('feature')['MI']

# OLS p-value (lower = more significant)
ranking['ols_pval']      = coef_df['pval']

# Granger p-value
ranking['granger_pval']  = gc_df.set_index('feature')['min_pval']

# Cointegration p-value
ranking['coint_pval']    = coint_df.set_index('feature')['pval']

# Composite score: average rank across positive metrics
rank_cols = ['pearson_abs', 'ccf_peak', 'rolling_mean', 'MI']
for col in rank_cols:
    ranking[f'{col}_rank'] = ranking[col].rank(ascending=False)
ranking['composite_rank'] = ranking[[f'{c}_rank' for c in rank_cols]].mean(axis=1)
ranking = ranking.sort_values('composite_rank')

display(ranking)

In [ ]:
# Save feature ranking
os.makedirs('output/reports', exist_ok=True)
ranking.to_csv('output/reports/02_feature_ranking.csv')
print('Saved: output/reports/02_feature_ranking.csv')

# Top features to carry forward into modeling
top_features = ranking.head(10).index.tolist()
print(f'\nTop 10 features for modeling: {top_features}')

## Next Steps

- Use `top_features` list as the starting feature set in `03_technical.ipynb`
- Drop features with VIF > 10 or OLS p > 0.05 if multicollinearity is severe
- If Johansen rank > 0, consider VECM as an additional benchmark model in `04_modeling.ipynb`